In [64]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from phd_helpers.AbaqusPostprocessing import (
    inp2pv, get_field_path, get_field_df, add_field_to_mesh, get_history_path, parse_final_step_time
)

In [92]:
def get_run_info(study, param_path):
     return {
            'study': study,
            'run_id': param_path.with_suffix('').name
        }

def get_nested(d, key, default=None):
    for part in key.split("."):
        d = d.get(part, default) if isinstance(d, dict) else default
    return d

def get_run_params(param_path, loop_params):

        with open(param_path, 'r') as f:
            params = json.load(f)

        return {k: get_nested(params, k, np.nan) for k in loop_params}
    
def get_run_results(path, run_id, sub, bone, run_id_mesh, pose, step, frame, field_metrics):

    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get meshes
    meshes = inp2pv(inp_file)
    mesh = meshes[bone]
    instance = f"{bone.upper()}_INST"

    # Field data
    for metric in field_metrics:
        field_path = get_field_path(csv_dir, metric, step, frame, instance)
        field_df = get_field_df(field_path)
        add_field_to_mesh(mesh, field_df)

    # History data
    history_data = pd.read_csv(get_history_path(csv_dir, step))
    RF_data = history_data[history_data['historyOutputKey']=='RF1']
    CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']

    # step data
    sta_path = inp_file.with_suffix('.sta')

    return {
        'RF': np.abs(RF_data['value'].iloc[frame]),
        'CA': CAREA_data['value'].iloc[frame],
        'P_max': mesh['CPRESS'].max(),
        'P_avg': np.mean(mesh['CPRESS'][mesh['CPRESS']>0]),
        #'loc_Pmax': mesh.points[np.argmax(mesh['CPRESS'])],
        'd' : parse_final_step_time(sta_path)
    }

def get_run_results_mesh(path, run_id, sub, bone, run_id_mesh, pose, step, frame, field_metrics):

    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get meshes
    meshes = inp2pv(inp_file)
    mesh = meshes[bone]
    instance = f"{bone.upper()}_INST"

    # Field data
    for metric in field_metrics:
        field_path = get_field_path(csv_dir, metric, step, frame, instance)
        field_df = get_field_df(field_path)
        add_field_to_mesh(mesh, field_df)

    return mesh


In [113]:
# input data
inp_dict = {
    'sub': '14548R',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field_metrics': ["CPRESS", 'U'],
    #'history_metrics': ['CAREA', 'RF']
}

# list of investigated parameters
loop_params = [
    'bone_material.E', 'bone_material.nu', 
    'cartilage_friction', 'cartilage_material.model', 'cartilage_material.D1',  'cartilage_material.C10',
    ]

studies = [1, 2, 3]

data = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_paths = path.glob('params/loop_params/*.json')

    for param_path in param_paths:
        info_dict = get_run_info(study, param_path)
        param_dict = get_run_params(param_path, loop_params)
        result_dict = get_run_results(path=path, run_id=info_dict['run_id'], **inp_dict)

        data.append(info_dict | param_dict | result_dict)


df = pd.DataFrame(data).sort_values(['study', 'run_id']).reset_index(drop=True)

In [114]:
df

,study,run_id,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.model,cartilage_material.D1,cartilage_material.C10,RF,CA,P_max,P_avg,d
0,1,00,1500,0.25,0.0,neo_hookean,0.0,0.085,50.007751,52.412712,2.589318,0.793261,0.391
1,1,01,1500,0.25,1.0,neo_hookean,0.0,0.085,49.998531,52.023518,2.589264,0.791678,0.388
2,1,02,1500,0.25,0.0,neo_hookean,1.0,0.085,50.007938,63.185146,2.940717,0.750057,0.702
3,1,03,1500,0.25,1.0,neo_hookean,1.0,0.085,26.284254,52.088802,1.428751,0.460954,0.522
4,1,04,1500,0.25,0.0,neo_hookean,0.0,0.110,49.998020,48.540871,2.775532,0.865280,0.360
5,1,05,1500,0.25,1.0,neo_hookean,0.0,0.110,49.998798,48.237209,2.783584,0.864025,0.357
6,1,06,1500,0.25,0.0,neo_hookean,1.0,0.110,50.044415,60.701347,2.957613,0.788268,0.679
7,1,07,1500,0.25,1.0,neo_hookean,1.0,0.110,36.549549,55.237282,1.978922,0.607434,0.585
8,1,08,1500,0.45,0.0,neo_hookean,0.0,0.085,49.998627,52.399796,2.590512,0.793674,0.390
9,1,09,1500,0.45,1.0,neo_hookean,0.0,0.085,50.019196,52.012440,2.591522,0.792199,0.387


In [115]:
def compute_avg_change(df_clean, metric):
    return df_clean.groupby(param)[metric].mean().diff().iloc[1]


metric = 'P_max'

df50 = df[df['RF'] > 49] # remove any runs that failed before 50 N
changes = []
for param in loop_params:
    df_clean = df50[~df50[param].isna()] # remove any runs that didn't use that parameter

    changes.append({
        'param': param,
        'change': compute_avg_change(df_clean, metric)
    })
pd.DataFrame(changes)

,param,change
0,bone_material.E,0.020425
1,bone_material.nu,0.003673
2,cartilage_friction,-0.158661
3,cartilage_material.model,0.197814
4,cartilage_material.D1,0.322356
5,cartilage_material.C10,0.131866


In [116]:
def paired_parameter_sensitivity(data, parameters, metric):
    """
    df : pandas.DataFrame
        Sensitivity study results.
    parameters : list[str]
        Input parameters to investigate.
    metric : str
        Output metric to compare, e.g. "P_max", "CA", "RF".
    """

    results = []

    df = data.copy()
    for param in parameters:
        levels = sorted(df[param].dropna().unique())

        if len(levels) != 2:
            raise ValueError(
                f"{param} has {len(levels)} levels, expected exactly 2: {levels}"
            )

        low, high = levels

        matching_cols = [p for p in parameters if p != param]
        if param == 'cartilage_material.model':
            matching_cols = [x for x in matching_cols if x!='cartilage_material.C10']
            df = data[data['cartilage_material.C10']!=0.085]

        low_df = df[df[param] == low].copy()
        high_df = df[df[param] == high].copy()

        merged = pd.merge(
            low_df,
            high_df,
            on=matching_cols,
            suffixes=("_low", "_high"),
        )

        changes = merged[f"{metric}_high"] - merged[f"{metric}_low"]


        results.append({
            "parameter": param,
            "low_value": low,
            "high_value": high,
            "n_pairs": len(changes),
            "max_change": changes.max(),
            "mean_change": changes.mean(),
            "min_change": changes.min()
        })

        if param == 'cartilage_material.model':
            df = data.copy()

    return pd.DataFrame(results)

In [ ]:
df50 = df[df['RF'] > 49] # remove any runs that failed before 50 N
summary = paired_parameter_sensitivity(df50, loop_params, "P_max")
summary

# could report % change relative to accuracy box results

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1500,3000,18,0.044729,0.020425,0.009660
1,bone_material.nu,0.25,0.45,18,0.013657,0.003673,0.000772
2,cartilage_friction,0.0,1.0,12,0.010059,0.003356,-0.001324
3,cartilage_material.model,neo_hookean,ogden,12,0.285814,0.131881,0.053132
4,cartilage_material.D1,0.0,1.0,12,0.426082,0.324034,0.182081
5,cartilage_material.C10,0.085,0.11,12,0.196567,0.131866,0.009487


In [ ]:
for what param combos are the sensitive params most sensitive
 - or which param combos result in change > some chosen value

Redo compression variation with constant tension behaviour

In [ ]:
figure out aire 
 - 1 .inp file ready to run on $SCRATCH/abaqus/testing
 - batch of 1 .inp file is ready to run on $SCRATCH/abaqus/testing/batch

Use abaqus cae and see what the input files look like for material settings and stuff 

## Magnitude of tensile strain

In [99]:
def get_field_df_wrap(sub, study, run_id, run_id_mesh, pose, bone, step, frame, field):
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 
    instance = f"{bone.upper()}_INST"
    
    field_path = get_field_path(csv_dir, field, step, frame, instance)
    return get_field_df(field_path)


inp_dict = {
    'sub': '14548R',
    'study': 2,
    'run_id': '00',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field': 'LE',
}
strain = get_field_df_wrap(**inp_dict)

In [107]:
def tensile_log_strain_array(df):
    n = len(df)

    LE = np.zeros((n, 3, 3), dtype=float)

    LE[:, 0, 0] = df["LE1"]
    LE[:, 1, 1] = df["LE2"]
    LE[:, 2, 2] = df["LE3"]

    LE[:, 0, 1] = LE[:, 1, 0] = df["LE4"]
    LE[:, 0, 2] = LE[:, 2, 0] = df["LE5"]
    LE[:, 1, 2] = LE[:, 2, 1] = df["LE6"]

    principal = np.linalg.eigvalsh(LE)
    max_principal = principal[:, -1]

    return np.maximum(max_principal, 0.0)

principal_tensile_strain = tensile_log_strain_array(strain)
print(f' Max: {principal_tensile_strain.max():.2f}')
print(f'Mean: {principal_tensile_strain.mean():.2f}')
print(f' 90%: {np.percentile(principal_tensile_strain, 90):.2f}')

 Max: 1.31
Mean: 0.14
 90%: 0.53
